# 📊 Model Evaluation — Retail Demand Forecasting

Detailed evaluation: residual analysis, feature importance, prediction distribution.

**Reads from:** `gold_ml_features`, saved model

**Writes to:** `gold_ml_feature_importance`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, abs as spark_abs, avg, when, isnan
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressionModel

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
model = RandomForestRegressionModel.load('Files/models/demand_forecasting_rf')
print('Model loaded')

In [ ]:
# Prepare test data (same pipeline as training)
features_df = spark.read.table('gold_ml_features')

# Clean nulls / NaN to match training
for c, dtype in features_df.dtypes:
    if dtype in ('double', 'float'):
        features_df = features_df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        features_df = features_df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

# Same feature set as training — EXCLUDE leakage (daily_revenue, transaction_count)
numeric_features = [
    'avg_discount', 'avg_price',
    'day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend',
    'demand_lag_1', 'demand_lag_7', 'revenue_lag_1',
    'unit_cost', 'margin_pct',
]

cat_cols = ['category', 'subcategory', 'region', 'store_format']
indexed_df = features_df
cat_idx_cols = []
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)
    cat_idx_cols.append(idx_col)

all_features = numeric_features + cat_idx_cols
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df)

_, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
predictions = model.transform(test_df)
print(f'Test predictions: {predictions.count():,} rows')

In [ ]:
# Residual analysis
residuals = (
    predictions
    .withColumn('residual', col('daily_quantity') - col('prediction'))
    .withColumn('abs_residual', spark_abs(col('residual')))
    .withColumn('pct_error',
        spark_abs(col('residual')) / col('daily_quantity') * 100
    )
)

print('=== Residual Statistics ===')
residuals.select('residual', 'abs_residual', 'pct_error').describe().show()

mape = residuals.filter(col('daily_quantity') > 0).agg(avg('pct_error')).collect()[0][0]
print(f'MAPE: {mape:.2f}%')

In [ ]:
# Feature importance from the RandomForest model (Spark stores it as a sparse vector).
# Build the Delta table from plain Python tuples (no pandas) for Fabric robustness.
importances = model.featureImportances.toArray()
rows = sorted(
    zip(all_features, [float(importances[i]) if i < len(importances) else 0.0
                       for i in range(len(all_features))]),
    key=lambda r: r[1], reverse=True,
)

print('=== Top 10 Features by Importance ===')
for name, imp in rows[:10]:
    print(f'  {name:30s} {imp:.4f}')

fi_spark = (
    spark.createDataFrame(rows, ['feature', 'importance'])
    .withColumn('model_timestamp', current_timestamp())
)
fi_spark.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_feature_importance')
print('Feature importance saved to gold_ml_feature_importance')

In [ ]:
spark.sql('OPTIMIZE gold_ml_feature_importance')
print('Evaluation complete')